In [102]:
import pandas as pd
final_design = pd.read_csv("/Users/alihashim/Desktop/Online_Academic_Submissions/poec_thesis/design_matrix/big_af_branch_design_final_corrected.txt", header=0)

In [156]:
import pandas as pd
import numpy as np

def print_unique_values(df):
    """
    Prints each column's name and its unique values, handling missing values.
    
    Parameters:
    df (pd.DataFrame): The input DataFrame.
    """
    for col in df.columns:
        unique_values = df[col].unique()  # Get unique values
        print(f"Column: {col}")
        print("Unique values:")
        print(unique_values)
        print("-" * 40)
        
    print_unique_values(final_design)

def get_scenario_submatrix(df_final, e_val, x_val, y_val, z_val):
    """
    Subset the final design for scenario (E=e_val, X=x_val, Y=y_val, Z=z_val).
    No expansions or merges—just a row filter.
    """
    sub_df = df_final[
        (df_final["E"] == e_val) &
        (df_final["X"] == x_val) &
        (df_final["Y"] == y_val) &
        (df_final["Z"] == z_val)
    ].copy()
    return sub_df


In [ ]:
# EPXANSIONS

import pandas as pd
from itertools import product

#######################
# 1) LOAD TOP-LEVEL CSV
#######################
df_top = pd.read_csv("/Users/alihashim/Desktop/Online_Academic_Submissions/poec_thesis/final_final_final_top_level_design.txt", header=0)  
# Must contain columns: A,B,G,H,I,E,X,Y,Z
# Example factor levels: A in {L1,L2,L3}, E in {L1,L2,L3}, X in {L1,L2}, etc.

#######################
# 2) SUB-FACTOR DEFINITIONS
#######################
# 2.1) Dimension-based sub-factors C,D,F
# We'll define a dictionary for each dimension
# Example: 3 levels for C, 3 for D, 4 for F.
# dimension=1 => label "C1_1"..."C1_3", "D1_1"..."D1_3", "F1_1"... "F1_4"
# dimension=2 => label "C2_1"..."C2_3", etc.
# dimension=3 => label "C3_..."

c_levels = {
  "L1": ["C1_1","C1_2","C1_3"],
  "L2": ["C2_1","C2_2","C2_3"],
  "L3": ["C3_1","C3_2","C3_3"]
}
d_levels = {
  "L1": ["D1_1","D1_2","D1_3"],
  "L2": ["D2_1","D2_2","D2_3"],
  "L3": ["D3_1","D3_2","D3_3"]
}
f_levels = {
  "L1": ["F1_1","F1_2","F1_3"],
  "L2": ["F2_1","F2_2","F2_3"],
  "L3": ["F3_1","F3_2","F3_3"]
}
# 2.2) N => 4 levels if Z=1, dimension-based
n_levels = {
  "L1": ["N1_1","N1_2"],
  "L2": ["N2_1","N2_2"],
  "L3": ["N3_1","N3_2"]
}

# 2.3) L,M => each 4 levels, not dimension-based, only if Z=1
L_4levels = ["L_1","L_2","L_3"]
M_4levels = ["M_1","M_2"]

# 2.4) J => 4 levels if X=1
J_4levels = ["L1","L2","L3"]  # or ["J0","J1","J2","J3"] if you prefer

# 2.5) K => 4 levels if Y=1
K_4levels =  ["L1","L2","L3"]


#######################
# 3) CARTESIAN PRODUCT HELPER
#######################
def cartesian(dfA, dfB):
    """Return the cartesian product of dfA x dfB."""
    dfA2 = dfA.assign(_tmpkey=1)
    dfB2 = dfB.assign(_tmpkey=1)
    out = pd.merge(dfA2, dfB2, on="_tmpkey").drop("_tmpkey",axis=1)
    return out

def list_to_df(colname, the_list):
    """Convert e.g. ["C1_1","C1_2"] to a single-col df with that column name."""
    return pd.DataFrame({colname: the_list})

#######################
# 4) BUILD THE FULL DESIGN
#######################
all_scenarios = []

for idx, row in df_top.iterrows():
    # Extract the top-level factor levels
    A_val = row["A"]
    B_val = row["B"]
    G_val = row["G"]
    H_val = row["H"]
    I_val = row["I"]
    E_val = row["E"]   # e.g. "L1","L2","L3"
    X_val = row["X"]   # e.g. "L1","L2"
    Y_val = row["Y"]
    Z_val = row["Z"]

    # We'll start with a 1-row df that has these top-level columns:
    df_base = pd.DataFrame([{
        "A":A_val,"B":B_val,"G":G_val,"H":H_val,"I":I_val,
        "E":E_val,"X":X_val,"Y":Y_val,"Z":Z_val
    }])

    # Next expand dimension-based sub-factors C,D,F
    c_list = c_levels[E_val]  # e.g. ["C1_1","C1_2","C1_3"] if E_val="L1"
    d_list = d_levels[E_val]
    f_list = f_levels[E_val]

    df_c = list_to_df("C", c_list)
    df_d = list_to_df("D", d_list)
    df_f = list_to_df("F", f_list)
    # Cartesian them all at once
    df_cdf = cartesian(cartesian(df_c, df_d), df_f)  # e.g. 3x3x4=36 combos

    # Now combine with df_base
    df_dim = cartesian(df_base, df_cdf)

    # If Z= "L2", we also have N plus L,M
    if Z_val=="L2":
        n_list = n_levels[E_val]  # 4 levels dimension-based
        df_n = list_to_df("N", n_list)
        df_dim = cartesian(df_dim, df_n)

        # L,M => each 4 levels, not dimension-based
        df_l = list_to_df("L", L_4levels)
        df_m = list_to_df("M", M_4levels)
        df_lm = cartesian(df_l, df_m)  # 4x4=16 combos
        df_dim = cartesian(df_dim, df_lm)
    # else if Z=L1 => no N,L,M expansions

    # Next expansions for (X,Y) => J,K
    # If X=L1,Y=L1 => no expansions
    # If X=L2,Y=L1 => cross with dfJ => 4 combos
    # If X=L1,Y=L2 => cross with dfK => 4 combos
    # If X=L2,Y=L2 => cross with dfJK => 16 combos
    # We'll do it after the dimension expansions are appended

    if X_val=="L2" and Y_val=="L1":
        # cross with J
        df_j = list_to_df("J", J_4levels)
        df_dim = cartesian(df_dim, df_j)
    elif X_val=="L1" and Y_val=="L2":
        # cross with K
        df_k = list_to_df("K", K_4levels)
        df_dim = cartesian(df_dim, df_k)
    elif X_val=="L2" and Y_val=="L2":
        # cross with J,K => 16 combos
        df_j = list_to_df("J", J_4levels)
        df_k = list_to_df("K", K_4levels)
        df_jk = cartesian(df_j, df_k)
        df_dim = cartesian(df_dim, df_jk)
    # else if X=L1,Y=L1 => do nothing

    # This df_dim is the final expansions for that top-level row
    # We collect them
    all_scenarios.append(df_dim)

# Combine all expansions
df_final = pd.concat(all_scenarios, ignore_index=True)

# Save or display
print("Final design shape:", df_final.shape)
df_final.to_csv("full_branched_design.csv", index=False)
print("Saved to full_branched_design.csv")


Final design shape: (519264, 17)
Saved to full_branched_design.csv


In [ ]:
# TESTING

import pandas as pd
import numpy as np

def check_design_integrity(df):
    """
    df must have columns: 
      [A,B,G,H,I,E,X,Y,Z, C, D, F, N, L, M, J, K] 
    with dimension-labeled expansions e.g. C1_,C2_,C3_ for E=L1,2,3, etc.
    We'll group by (E,X,Y,Z) and examine sub-factor coverage.
    """

    # group by scenario
    scenario_counts = df.groupby(["E","X","Y","Z"]).size().reset_index(name="Count")

    # sub-factors in an ideal scenario
    # dimension-labeled expansions
    # We'll define a helper to see if a string matches dimension i
    def is_dimC(value, eVal):
        # eVal in {"L1","L2","L3"}
        # check if value starts with e.g. "C1_"
        if pd.isna(value):
            return value
        return (str(value).startswith("C1_") and eVal=="L1") or \
               (str(value).startswith("C2_") and eVal=="L2") or \
               (str(value).startswith("C3_") and eVal=="L3")

    # We'll define a dictionary that tells us how many expansions we expect
    # for each dimension-labeled sub-factor if it is present.
    expected_levels = {
      "C": 3,  # each dimension => 3 combos
      "D": 3,
      "F": 4,
      "N": 4  # only if Z=1
    }
    # L,M => each 4 levels if Z=1
    # J,K => each 4 levels if X=1 or Y=1

    # We'll do a scenario-level check:
    for idx,row in scenario_counts.iterrows():
        eVal = row["E"]
        xVal = row["X"]
        yVal = row["Y"]
        zVal = row["Z"]
        count = row["Count"]

        # subset
        scenario_df = df[(df["E"]==eVal)&(df["X"]==xVal)&(df["Y"]==yVal)&(df["Z"]==zVal)]
        scenario_size = len(scenario_df)

        print(f"\n=== Scenario (E={eVal},X={xVal},Y={yVal},Z={zVal}) => {count} rows ===")
        if scenario_size!= count:
            print("  WARNING: subset size mismatch??")

        # check dimension-labeled columns C,D,F:
        # they should match dimension eVal. 
        # e.g. if eVal="L1", C column must contain only C1_*
        if not scenario_df.empty:
            # We'll look at the distinct values in each sub-factor
            distinctC = scenario_df["C"].dropna().unique()
            # confirm each is consistent
            # we'll do a small check
            inconsistentC = [v for v in distinctC if not is_dimC(v,eVal)]
            if inconsistentC:
                print(f"  WARNING: Found dimension mismatch in C => {inconsistentC}")
            # also check the number of distinct expansions
            # if eVal="L1" and we see 3 distinct expansions => good
            # but only if scenario_df actually has coverage for them
            print(f"  Distinct C in scenario => {distinctC}")

            # Repeat for D,F with similar checks
            # We'll define small internal checks for D,F
            # skipping full code detail for brevity - concept is the same

            # N => only if zVal="L2"
            if zVal=="L2":
                distinctN = scenario_df["N"].dropna().unique()
                # check dimension prefix
                # ...
                print(f"  Distinct N => {distinctN}")
            else:
                # ensure N is all NaN
                non_na_n = scenario_df["N"].dropna()
                if len(non_na_n)>0:
                    print("  WARNING: Found N in Z=0 scenario??")

            # L,M => only if zVal="L2", each 4 levels
            if zVal=="L2":
                distinctL = scenario_df["L"].dropna().unique()
                distinctM = scenario_df["M"].dropna().unique()
                print(f"  Distinct L => {distinctL}")
                print(f"  Distinct M => {distinctM}")
                # check they have exactly 4 levels or coverage
            else:
                # ensure L,M are empty
                pass

            # J => only if xVal=="L2"
            if xVal=="L2":
                distinctJ = scenario_df["J"].dropna().unique()
                print(f"  Distinct J => {distinctJ}")
            else:
                # check all J are NaN
                pass

            # K => only if yVal=="L2"
            if yVal=="L2":
                distinctK = scenario_df["K"].dropna().unique()
                print(f"  Distinct K => {distinctK}")
            else:
                # check all K are NaN
                pass

    print("\n=== Diagnostics complete ===")


### Usage Example

if __name__=="__main__":
    df_design = pd.read_csv("full_branched_design.csv") 
    # This design should have columns 
    # [A,B,G,H,I,E,X,Y,Z,C,D,F,N,L,M,J,K]
    check_design_integrity(df_design)


In [168]:
get_scenario_submatrix(df_final, "L1", "L2", "L2", "L2")

,A,B,G,H,I,E,X,Y,Z,C,D,F,J,K,N,L,M
17550,L2,L3,L3,L3,L1,L1,L2,L2,L2,C1_1,D1_1,F1_1,L1,L1,N1_1,L_1,M_1
17551,L2,L3,L3,L3,L1,L1,L2,L2,L2,C1_1,D1_1,F1_1,L1,L2,N1_1,L_1,M_1
17552,L2,L3,L3,L3,L1,L1,L2,L2,L2,C1_1,D1_1,F1_1,L1,L3,N1_1,L_1,M_1
17553,L2,L3,L3,L3,L1,L1,L2,L2,L2,C1_1,D1_1,F1_1,L2,L1,N1_1,L_1,M_1
17554,L2,L3,L3,L3,L1,L1,L2,L2,L2,C1_1,D1_1,F1_1,L2,L2,N1_1,L_1,M_1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
497362,L1,L1,L2,L1,L2,L1,L2,L2,L2,C1_3,D1_3,F1_3,L2,L2,N1_2,L_3,M_2
497363,L1,L1,L2,L1,L2,L1,L2,L2,L2,C1_3,D1_3,F1_3,L2,L3,N1_2,L_3,M_2
497364,L1,L1,L2,L1,L2,L1,L2,L2,L2,C1_3,D1_3,F1_3,L3,L1,N1_2,L_3,M_2
497365,L1,L1,L2,L1,L2,L1,L2,L2,L2,C1_3,D1_3,F1_3,L3,L2,N1_2,L_3,M_2


In [170]:
# level_to_desc = {
    
#     "issue" : {"L1": 1, "L2":2, "L3":3 },
#     "static_cand": {"L1":"dynamic_cand" , "L2":"static_cand"},
#     "exo_party": {"L1":"", "L2":}
    
    
    
# }

for issue_level in ["L1", "L2", "L3"]:

    for static_cand in ["L1", "L2"]:
    
        for exo_party in ["L1", "L2"]:
            
            for imperf_voter in ["L1", "L2"]:
                
                df = get_scenario_submatrix(df_final, issue_level, static_cand, exo_party, imperf_voter)
                df.to_csv(f"issue_{issue_level}_cand_{static_cand}_party_{exo_party}_voter_{imperf_voter}.csv")
                
            
        
        